# Results for model: thudm/chatglm3-6b

In [ ]:
import os
import pandas as pd
import polars as polars
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import XGBRegressor
from sklearn.metrics import mean_squared_error

# Load the data
data_path = './jane_street_data/train.parquet'
df = polars.read(data_path)

# Perform feature engineering: calculate the top 15% of 'feature_00' relative to 'responder_6'
top_quantile = (df['feature_00'] >= (df[' feature_00'].quantile(0.875) - 0.15).min())
df['top_quantile'] = top_quantile

# Split the data into train and test sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Create transformers for feature engineering
num_features = ['feature_00']
cat_features = ['feature_01']

num_transformer = ColumnTransformer(
    transformers=[
        ('num',NUM_SERIALIZED_NUMERIC, num_features),
        ('cat',CATEGORICAL_SERIALIZED, cat_features)
    ])

cat_transformer = ColumnTransformer(
    transformers=[
        ('cat', CATEGORICAL_SERIALIZED, cat_features),
        ('any', fa.Any(), ['feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15']),
        ('any', fa.Any(), ['feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20']),
        ('any', fa.Any(), ['date_id', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25']),
        ('onehot', OneHotEncoder(handle_unknown='ignore').fit_transform, cat_features)
    ])

preprocessor_steps = [('num_transformer', num_transformer),
                     ('cat_transformer', cat_transformer)]

# Create XGBoost pipeline
xgb_pipeline = Pipeline([('preprocessor', preprocessor_steps),
                         ('xgb_regr', XGBRegressor())])

# Train the model
xgb_pipeline.fit(train_df.drop('date_id', axis=1),
                  train_df['date_id'],
                  test_size=0.2,
                  epochs=100,
                  verbose=0,
                  n_jobs=-1)

# Evaluate the model
y_true = test_df['responder_6']
y_pred = xgb_pipeline.predict(test_df.drop('date_id', axis=1))
mse = mean_squared_error(y_true, y_pred)
print("MSE: %.3f" % mse)